### Data Pipiline: Data Ingestion to Vector DB Store

In [1]:
import os
from pathlib import Path  # noqa: F401

from langchain_community.document_loaders import (
    DirectoryLoader,
    PyMuPDFLoader,
    PyPDFLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)




/var/folders/xs/r6p2mb494tg5q69l24s38t0c0000gn/T/ipykernel_17120/4046972036.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdfs")

Found 3 PDF files to process

Processing: Guide-to-Boston.pdf
  ✓ Loaded 6 pages

Processing: Freedom Trail Brochure 2025.pdf
  ✓ Loaded 28 pages

Processing: MTAI-v1i1-paper-02.pdf
  ✓ Loaded 16 pages

Total documents loaded: 50


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m119 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Guide to Boston', 'source': '../data/pdfs/Guide-to-Boston.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Guide-to-Boston.pdf', 'file_type': 'pdf'}, page_content='IOC\nGUIDE\nTO\nBOSTON\nTHINGS\nTO\nDO\nMuseums\n●\nMuseum\nof\nFine\nArts\n–\nOne\nof\nthe\nmost\ncomprehensive\nart\nmuseums\nin\nthe\nworld,\nthe\nMFA\ncollection\nencompasses\nalmost\n500,000\nworks\nof\nart.\nFeatured\ngalleries\ninclude\nMonet,\nFrida\nKahlo,\nand\nToulouse-Lautrec\nand\nthe\nStars\nof\nParis.\nConveniently\nlocated\noutside\nof\nthe\nFenway\narea.\n●\nIsabella\nStewart\nGardner\nMuseum\n–\nIn\n1990,\nthe\nsingle\nlargest\nproperty\ntheft\nin\nthe\nworld\noccurred\nat\nthis\nmuseum\nstarted\nby\nIsabella\nStewart\nGardner\nand\nher\nhusband.\nIt\nremains\nthe\nbiggest\nunsolved\nart\ntheft\nin\nworld\nhistory.\nAlso\nlocated\noutside\nof\nthe\nFenway\narea\nnear\n

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=500,chunk_overlap=100):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 50 documents into 306 chunks

Example chunk:
Content: IOC
GUIDE
TO
BOSTON
THINGS
TO
DO
Museums
●
Museum
of
Fine
Arts
–
One
of
the
most
comprehensive
art
museums
in
the
world,
the
MFA
collection
encompasses
almost
500,000
works
of
art.
Featured
galleries
...
Metadata: {'producer': 'Skia/PDF m119 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Guide to Boston', 'source': '../data/pdfs/Guide-to-Boston.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Guide-to-Boston.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m119 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Guide to Boston', 'source': '../data/pdfs/Guide-to-Boston.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Guide-to-Boston.pdf', 'file_type': 'pdf'}, page_content='IOC\nGUIDE\nTO\nBOSTON\nTHINGS\nTO\nDO\nMuseums\n●\nMuseum\nof\nFine\nArts\n–\nOne\nof\nthe\nmost\ncomprehensive\nart\nmuseums\nin\nthe\nworld,\nthe\nMFA\ncollection\nencompasses\nalmost\n500,000\nworks\nof\nart.\nFeatured\ngalleries\ninclude\nMonet,\nFrida\nKahlo,\nand\nToulouse-Lautrec\nand\nthe\nStars\nof\nParis.\nConveniently\nlocated\noutside\nof\nthe\nFenway\narea.\n●\nIsabella\nStewart\nGardner\nMuseum\n–\nIn\n1990,\nthe\nsingle\nlargest\nproperty\ntheft\nin\nthe\nworld\noccurred\nat\nthis\nmuseum\nstarted\nby\nIsabella\nStewart\nGardner\nand\nher\nhusband.\nIt'),
 Document(metadata={'producer': 'Skia/PDF m119 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title'

### Embedding And vectorStoreDB

In [ ]:
import uuid
import chromadb
import numpy as np
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from typing import Any


/Users/rajeshlingam/gitViews/bu_rag_workshop/BURAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 21:57:27.079000 17120 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9602.00it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store", reset: bool = False):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
            reset: Delete any existing collection first (avoids duplicates on re-runs)
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.reset = reset
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            if self.reset:
                try:
                    self.client.delete_collection(self.collection_name)
                except Exception:
                    pass  # collection did not exist yet

            # Get or create collection (cosine so that similarity = 1 - distance)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG", "hnsw:space": "cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore(reset=True)
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [9]:
chunks

[Document(metadata={'producer': 'Skia/PDF m119 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Guide to Boston', 'source': '../data/pdfs/Guide-to-Boston.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Guide-to-Boston.pdf', 'file_type': 'pdf'}, page_content='IOC\nGUIDE\nTO\nBOSTON\nTHINGS\nTO\nDO\nMuseums\n●\nMuseum\nof\nFine\nArts\n–\nOne\nof\nthe\nmost\ncomprehensive\nart\nmuseums\nin\nthe\nworld,\nthe\nMFA\ncollection\nencompasses\nalmost\n500,000\nworks\nof\nart.\nFeatured\ngalleries\ninclude\nMonet,\nFrida\nKahlo,\nand\nToulouse-Lautrec\nand\nthe\nStars\nof\nParis.\nConveniently\nlocated\noutside\nof\nthe\nFenway\narea.\n●\nIsabella\nStewart\nGardner\nMuseum\n–\nIn\n1990,\nthe\nsingle\nlargest\nproperty\ntheft\nin\nthe\nworld\noccurred\nat\nthis\nmuseum\nstarted\nby\nIsabella\nStewart\nGardner\nand\nher\nhusband.\nIt'),
 Document(metadata={'producer': 'Skia/PDF m119 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title'

In [10]:
## Convert text chunks to embeedings
texts = [chunk.page_content for chunk in chunks]

## Generate Embeedings
embeddings = embedding_manager.generate_embeddings(texts)

## Store  in the vector store
vectorstore.add_documents(chunks, embeddings)




Generating embeddings for 306 texts...


Batches: 100%|██████████| 10/10 [00:00<00:00, 13.18it/s]


Generated embeddings with shape: (306, 384)
Adding 306 documents to vector store...
Successfully added 306 documents to vector store
Total documents in collection: 306


### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("What is the Freedom Trail?", top_k=3, score_threshold=0.0)
## More Queries can be added here
# "Which museums can I visit in Boston?" (Guide)
# "Where does the Freedom Trail start?" (Brochure)
# "How long is the Freedom Trail?" (Brochure, about 2.5 miles)
# "What is Boston Common?" (both)

Retrieving documents for query: 'What is the Freedom Trail?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_a0980f3e_5',
  'content': 'history\nand\nthe\nstart\nof\nthe\nnation\nin\nthe\nlate\n1700s.\nVisitors\ncan\neither\nembark\non\nguided\ntours\nor\nfollow\nthe\nbrick\npathway\nthrough\nthe\ncity,\nstopping\nat\nsites\nat\ntheir\nown\npace.\nSome\nof\nthe\nmost\nnotable\nspots\non\nthe\nFreedom\ntrail\ninclude:',
  'metadata': {'producer': 'Skia/PDF m119 Google Docs Renderer',
   'creator': 'PyPDF',
   'title': 'Guide to Boston',
   'source_file': 'Guide-to-Boston.pdf',
   'content_length': 241,
   'source': '../data/pdfs/Guide-to-Boston.pdf',
   'page_label': '1',
   'file_type': 'pdf',
   'doc_index': 5,
   'page': 0,
   'creationdate': '',
   'total_pages': 6},
  'similarity_score': 0.7107414603233337,
  'distance': 0.28925853967666626,
  'rank': 1},
 {'id': 'doc_adcae369_38',
  'content': 'On behalf of the \nCommonwealth of \nMassachusetts, we welcome \nyou to the Freedom Trail. As \nwe begin our celebrations \nfor the 250th anniversary of \nthe American Revolution in \

In [ ]:
results = rag_retriever.retrieve("Where does the Freedom Trail start?", top_k=3)
for r in results:
    print(f"[{r['rank']}] score={r['similarity_score']:.3f} | {r['metadata']['source_file']} p.{r['metadata']['page']}")
    print(r['content'][:200].replace("\n", " "), "\n")

    #"What is the biggest unsolved art theft in history, and where did it happen?"
    #"What is the Freedom Trail and how many historic sites does it connect?"
    #"Which ship is nicknamed Old Ironsides?"
    #"Which museum has a butterfly garden and a planetarium?"
    #"What two models make up the proposed hybrid zero-trust framework?"
    #"Which baseline models was it compared against?"

Retrieving documents for query: 'Where does the Freedom Trail start?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.88it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
[1] score=0.738 | Guide-to-Boston.pdf p.0
history and the start of the nation in the late 1700s. Visitors can either embark on guided tours or follow the brick pathway through the city, stopping at sites at their own pace. Some of the most no 

[2] score=0.657 | Freedom Trail Brochure 2025.pdf p.2
On behalf of the  Commonwealth of  Massachusetts, we welcome  you to the Freedom Trail. As  we begin our celebrations  for the 250th anniversary of  the American Revolution in  Massachusetts, the Free 

[3] score=0.651 | Freedom Trail Brochure 2025.pdf p.1
they risked striving for victory and freedom. Walk Into History®  on the Freedom Trail with 18th-century costumed Freedom Trail  Players® or at your own pace. Step INSIDE the places where the  America 



### Integration Vectordb Context pipeline With LLM output


### Simple RAG pipeline with OpenAI LLM


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # also picks up OPENAI_API_KEY from a .env file if you use one

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1, max_tokens=1024)  # reads OPENAI_API_KEY from the environment

def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join(doc["content"] for doc in results) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer
    prompt = f"""Use the following context to answer the question concisely.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content




Retrieving documents for query: 'Which ship is nicknamed Old Ironsides?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
USS Constitution


In [17]:
answer=rag_simple("Which ship is nicknamed Old Ironsides?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Which ship is nicknamed Old Ironsides?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


USS Constitution


In [18]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is the Freedom Trail and how many historic sites does it connect?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the Freedom Trail and how many historic sites does it connect?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.88it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: The Freedom Trail is a 2.5-mile red line in Boston that connects 16 nationally significant historic sites, showcasing over 250 years of history.
Sources: [{'source': 'Guide-to-Boston.pdf', 'page': 0, 'score': 0.7498055100440979, 'preview': 'history\nand\nthe\nstart\nof\nthe\nnation\nin\nthe\nlate\n1700s.\nVisitors\ncan\neither\nembark\non\nguided\ntours\nor\nfollow\nthe\nbrick\npathway\nthrough\nthe\ncity,\nstopping\nat\nsites\nat\ntheir\nown\npace.\nSome\nof\nthe\nmost\nnotable\nspots\non\nthe\nFreedom\ntrail\ninclude:...'}, {'source': 'Freedom Trail Brochure 2025.pdf', 'page': 1, 'score': 0.7302727699279785, 'preview': 'Walk the Freedom Trail® through history!\nExperience more than 250 years of history along the Freedom \nTrail – Boston’s iconic 2.5 mile red line leading to 16 nationally \nsignificant historic sites. The Freedom Trail is a unique collection \nof museums, churches, meeting houses

In [19]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Places to visit in Boston?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


Retrieving documents for query: 'Places to visit in Boston?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.38it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Boston
University’s
campus.
Expert
tour
guides
will
take
visitors
through
the
park’s
rich
history
as
a
Boston
site
and
icon
of
American
baseball.
●
Duck
Boat
Tours
–
Hop
on
a
renovated
World
War
II
amphibious
vehicle
to
get
one
of
the
most
comprehensi

ve
tours
of
Boston
by
land
and
sea.
This
is
a
must-do
activity
for
any
first-time
visitor!
●
Walk
around
the
Boston
Harbor
and
North
End
○
If
the
weather
is
nice,
take
a
walk
down
the
Rose
Kennedy
Greenway,
a
contemporary
public
park
that
replaced

is
nice,
take
a
walk
down
the
Rose
Kennedy
Greenway,
a
contemporary
public
park
that
replaced
the
central
artery
highway
during
the
infamous
Big
Dig.
●
Take
a
photo
on
Acorn
Street
○
One
of
the
most
photographed
areas
in
the
city,
this
quaint
street
is
lined
with
quaint
row
houses
reminiscent
of
Boston’s
colonial
era.
●
Check
out
Meet
Boston
for
an
updated
calendar
of
events

13
National Parks of Boston – Boston National Historical Park
Faneuil Hall* Visitor Center
The Great Hall, Wednesdays–Sundays, 
10am-5pm
Visit website for current programs
nps.gov/bost/bost-know-before-you-go.htm
Ranger-led Black Heritage Trail® Tours
Wednesdays–Sundays, Reservations on 
Eventbrite required
Visit website for current programs
617-742-5415 
nps.gov/boaf/p